In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_10_account_events_mapped
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_10_account_events_mapped
# Spec Ref  : §1.6 Event Aggregation — Account-Level (mk_account_events_mapped)
# Purpose   : Aggregate events per account × meta_event × event_day.
#             Combines IDENTIFIED events (anonymous_activity=false)
#             and ANONYMOUS events (anonymous_activity=true).
#             This gives scoring models both signals per account.
#
# Identified: contactid is known → roll up through contacts_to_accounts
# Anonymous : contactid IS NULL → match event to account by domain
#
# Incremental MERGE — only reprocesses events since last aggregated day.
#
# Run after : map_04 (mapped_events), map_05 (contacts_to_accounts)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)
print(f"=== Map 10: mk_account_events_mapped (incremental MERGE) ===  customer={customer_id}  tenant={tenant_id}")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

initialize_audit_tables()

In [0]:
# CELL 3
create_map_table(
    f"{sv}.mk_account_events_mapped",
    """
        mk_accountid_domain  STRING  NOT NULL,
        meta_event           STRING  NOT NULL,
        event_day            DATE    NOT NULL,
        occurrences          BIGINT,
        anonymous_activity   BOOLEAN,
        tenant               BIGINT
    """,
    cluster_by="mk_accountid_domain, meta_event, event_day"
)

In [0]:
# CELL 4 ── CDF-Aware Incremental Aggregation for mk_account_events_mapped
# Uses CDF from mapped_events to find the minimum affected event_day,
# then re-aggregates only from that day forward.
try:
    # ── VERSION TRACKING ──
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {sv}.mk_account_events_mapped ('last_source_cdf_version')").collect()
        val = props[0][1] if props else None
        last_source_ver = int(val) if val and val != '(not specified)' and 'does not have property' not in str(val) else 0
    except Exception:
        last_source_ver = 0

    try:
        source_max_ver = spark.sql(f"SELECT MAX(version) FROM (DESCRIBE HISTORY {sv}.mapped_events)").collect()[0][0] or 0
    except Exception:
        source_max_ver = 0

    print(f"  CDF: source={sv}.mapped_events  last_read_ver={last_source_ver}  current_source_ver={source_max_ver}")

    target_count = spark.sql(f"SELECT COUNT(*) FROM {sv}.mk_account_events_mapped WHERE tenant = {tenant_id}").collect()[0][0]

    if target_count == 0:
        print(f"  First run — aggregating all events")
        agg_from_day = "1900-01-01"
    elif last_source_ver >= source_max_ver:
        print(f"  Source unchanged (ver {source_max_ver}) — skipping")
        agg_from_day = None
    else:
        start_ver = last_source_ver + 1
        try:
            from pyspark.sql import functions as F
            cdf_df = (spark.read.format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_ver)
                .table(f"{sv}.mapped_events")
                .filter(f"tenant = {tenant_id}")
                .filter("_change_type IN ('insert', 'update_postimage', 'delete')")
            )
            min_day_row = cdf_df.select(F.min(F.col("event_timestamp").cast("date")).alias("min_day")).collect()
            if min_day_row and min_day_row[0]["min_day"]:
                agg_from_day = str(min_day_row[0]["min_day"])
                cdf_count = cdf_df.count()
                print(f"  CDF: {cdf_count} changes, re-aggregating from {agg_from_day}")
            else:
                print(f"  CDF: no relevant changes for tenant {tenant_id}")
                agg_from_day = None
        except Exception as cdf_err:
            print(f"  CDF read failed ({cdf_err}), falling back to watermark")
            agg_from_day = str(spark.sql(f"""
                SELECT COALESCE(MAX(event_day), DATE('1900-01-01'))
                FROM {sv}.mk_account_events_mapped WHERE tenant = {tenant_id}
            """).collect()[0][0])

    if agg_from_day is not None:
        spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW account_events_delta AS

            -- IDENTIFIED: contact is known → roll up via contacts_to_accounts
            SELECT
                c2a.account_id                            AS mk_accountid_domain,
                e.meta_event,
                CAST(e.event_timestamp AS DATE)           AS event_day,
                COUNT(*)                                  AS occurrences,
                false                                     AS anonymous_activity,
                e.tenant
            FROM {sv}.mapped_events e
            INNER JOIN {sv}.contacts_to_accounts c2a
                ON e.contactid = c2a.contact_id
            WHERE e.contactid IS NOT NULL
              AND e.tenant = {tenant_id}
              AND CAST(e.event_timestamp AS DATE) >= '{agg_from_day}'
            GROUP BY
                c2a.account_id,
                e.meta_event,
                CAST(e.event_timestamp AS DATE),
                e.tenant

            UNION ALL

            -- ANONYMOUS: no contact resolved → match to account by domain
            SELECT
                a.id                                      AS mk_accountid_domain,
                e.meta_event,
                CAST(e.event_timestamp AS DATE)           AS event_day,
                COUNT(*)                                  AS occurrences,
                true                                      AS anonymous_activity,
                e.tenant
            FROM {sv}.mapped_events e
            INNER JOIN {sv}.accounts_all a
                ON LOWER(e.domain) = LOWER(a.domain)
            WHERE e.contactid IS NULL
              AND e.domain IS NOT NULL
              AND TRIM(e.domain) != ''
              AND e.tenant = {tenant_id}
              AND CAST(e.event_timestamp AS DATE) >= '{agg_from_day}'
            GROUP BY
                a.id,
                e.meta_event,
                CAST(e.event_timestamp AS DATE),
                e.tenant
        """)

        # Add tenant column if missing
        existing_cols = [c.name for c in spark.table(f"{sv}.mk_account_events_mapped").schema]
        if "tenant" not in existing_cols:
            spark.sql(f"ALTER TABLE {sv}.mk_account_events_mapped ADD COLUMN tenant BIGINT")

        spark.sql(f"""
            MERGE INTO {sv}.mk_account_events_mapped AS target
            USING account_events_delta AS source
            ON  target.mk_accountid_domain = source.mk_accountid_domain
            AND target.meta_event          = source.meta_event
            AND target.event_day           = source.event_day
            AND target.anonymous_activity  = source.anonymous_activity
            AND target.tenant              = source.tenant
            WHEN MATCHED THEN
                UPDATE SET target.occurrences = source.occurrences
            WHEN NOT MATCHED THEN
                INSERT (mk_accountid_domain, meta_event, event_day,
                        occurrences, anonymous_activity, tenant)
                VALUES (source.mk_accountid_domain, source.meta_event,
                        source.event_day, source.occurrences,
                        source.anonymous_activity, source.tenant)
        """)

    # Save version
    spark.sql(f"ALTER TABLE {sv}.mk_account_events_mapped SET TBLPROPERTIES ('last_source_cdf_version' = '{source_max_ver}')")
    print(f"  Saved last_source_cdf_version = {source_max_ver}")

    # Verify both activity types exist
    n     = cnt(f"{sv}.mk_account_events_mapped")
    anon  = spark.sql(f"SELECT COUNT(*) FROM {sv}.mk_account_events_mapped WHERE anonymous_activity = true AND tenant = {tenant_id}").collect()[0][0]
    ident = spark.sql(f"SELECT COUNT(*) FROM {sv}.mk_account_events_mapped WHERE anonymous_activity = false AND tenant = {tenant_id}").collect()[0][0]
    print(f"  mk_account_events_mapped: {n:,} rows (all tenants)")
    print(f"  Tenant {tenant_id} — Identified events: {ident:,}")
    print(f"  Tenant {tenant_id} — Anonymous events : {anon:,}")
    status = '\u2705' if anon > 0 and ident > 0 else '\u26a0 missing one type'
    print(f"  Both types present: {status}")
except Exception as e:
    print(f"\u274c mk_account_events_mapped processing failed: {e}")
    log_audit(
        job_name       = "02b_map_10_account_events_mapped",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise